> **Fixed version note**
>
> - Added a safer path lookup for `chinook.db`.
> - Kept the notebook logic intact because the original JOIN examples already ran correctly.
> - This file was validated with the uploaded local database.

# Lecture 2: SQL JOINs & Pandas Data Manipulation

In this lecture, we'll explore how to combine data from multiple tables using **SQL JOINs** and how to filter and manipulate DataFrames using **Pandas**. We'll learn the differences between `.loc` and `.iloc`, understand how to apply boolean masks for filtering, and discover important indexing patterns to avoid common pitfalls when working with filtered data.

### Install dependencies

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


We define a helper function `run_query()` to easily run SQL commands and return the results as a Pandas DataFrame.

### Import libraries

In [13]:
import pandas as pd
import sqlite3

In [3]:
from pathlib import Path
import pandas as pd
import sqlite3

# FIX: use a robust local path so the notebook works from the repo root
db_candidates = [Path('chinook.db'), Path('/mnt/data/chinook.db')]
db_path = next((p for p in db_candidates if p.exists()), None)
if db_path is None:
    raise FileNotFoundError("Could not find chinook.db in the working directory.")

db_filename = str(db_path)

def run_query(q):
    # This creates a connection, runs the query, and closes the connection automatically
    with sqlite3.connect(db_filename) as conn:
        return pd.read_sql(q, conn)

# Test the connection
try:
    print("Connecting to database...")
    tables = run_query("SELECT name FROM sqlite_master WHERE type='table';")
    print("Success! Tables found:")
    print(tables)
except Exception as e:
    print(f"Error: {e}")
    print("Double-check that 'chinook.db' is present and readable.")


Connecting to database...
Success! Tables found:
              name
0            album
1           artist
2         customer
3         employee
4            genre
5          invoice
6     invoice_line
7       media_type
8         playlist
9   playlist_track
10           track


### Database Tables
A database most often contains one or more tables. Each table is identified by a name (e.g., "album" or "artist") and contains records (rows) with data.

We can see the list of tables in our database below:

In [4]:
run_query("SELECT name FROM sqlite_master WHERE type='table';")

,name
0,album
1,artist
2,customer
3,employee
4,genre
5,invoice
6,invoice_line
7,media_type
8,playlist
9,playlist_track


### 1. JOIN
We can assign to each row an id and connect entities through their ids.

In [5]:
run_query(
"""SELECT
    artist.name AS Artist,
    album.title AS Album
FROM
    album
INNER JOIN
    artist ON album.artist_id = artist.artist_id;""").head()

,Artist,Album
0,AC/DC,For Those About To Rock We Salute You
1,Accept,Balls to the Wall
2,Accept,Restless and Wild
3,AC/DC,Let There Be Rock
4,Aerosmith,Big Ones


### 2. Different Types of SQL JOINs
Here are the different types of the JOINs in SQL:
* (INNER) JOIN: Returns records that have matching values in both tables
* LEFT (OUTER) JOIN: Returns all records from the left table, and the matched records from the right table
* RIGHT (OUTER) JOIN: Returns all records from the right table, and the matched records from the left table
* FULL (OUTER) JOIN: Returns all records when there is a match in either left or right table

#### LEFT JOIN
The LEFT JOIN keyword returns all records from the left table (table1), and the matching records from the right table (table2). If there is no match on table2 then it will present NULL values for table2 columns.


In [6]:
run_query(
"""SELECT
    artist.name as Artist,
    album.title as Album
FROM
    artist
LEFT JOIN
    album 
ON album.artist_id = artist.artist_id;""").head()

,Artist,Album
0,AC/DC,For Those About To Rock We Salute You
1,AC/DC,Let There Be Rock
2,Accept,Balls to the Wall
3,Accept,Restless and Wild
4,Aerosmith,Big Ones


In [7]:
run_query(
"""SELECT
    artist.name as Artist,
    album.title as Album
FROM
    artist
LEFT JOIN
    album 
ON album.artist_id = artist.artist_id
WHERE 
Album is NULL;""").head()

,Artist,Album
0,Milton Nascimento & Bebeto,None
1,Azymuth,None
2,João Gilberto,None
3,Bebel Gilberto,None
4,Jorge Vercilo,None


### 3. Multiple JOINS

We can join tables multiple times. This is useful when you need to combine data from 3 or more tables.

In this example, we join the **artist**, **album**, and **track** tables together:
1. First, we join `artist` with `album` using `artist_id`
2. Then, we join the result with `track` using `album_id`

This gives us artist names, album titles, and the tracks within each album.


In [8]:
run_query("""
SELECT
    artist.name AS Artist,
    album.title AS Album,
    track.name AS Track
FROM
    artist
JOIN
    album ON artist.artist_id = album.artist_id
JOIN
    track ON album.album_id = track.album_id
LIMIT 10""")


,Artist,Album,Track
0,AC/DC,For Those About To Rock We Salute You,For Those About To Rock (We Salute You)
1,Accept,Balls to the Wall,Balls to the Wall
2,Accept,Restless and Wild,Fast As a Shark
3,Accept,Restless and Wild,Restless and Wild
4,Accept,Restless and Wild,Princess of the Dawn
5,AC/DC,For Those About To Rock We Salute You,Put The Finger On You
6,AC/DC,For Those About To Rock We Salute You,Let's Get It Up
7,AC/DC,For Those About To Rock We Salute You,Inject The Venom
8,AC/DC,For Those About To Rock We Salute You,Snowballed
9,AC/DC,For Those About To Rock We Salute You,Evil Walks


You can also use multiple LEFT JOINs to include artists, albums, and genres even if some data is missing:


In [9]:
run_query("""
SELECT
    artist.name AS Artist,
    album.title AS Album,
    track.name AS Track,
    genre.name AS Genre
FROM
    artist
LEFT JOIN
    album ON artist.artist_id = album.artist_id
LEFT JOIN
    track ON album.album_id = track.album_id
LEFT JOIN
    genre ON track.genre_id = genre.genre_id
LIMIT 10""")


,Artist,Album,Track,Genre
0,AC/DC,For Those About To Rock We Salute You,For Those About To Rock (We Salute You),Rock
1,AC/DC,For Those About To Rock We Salute You,Put The Finger On You,Rock
2,AC/DC,For Those About To Rock We Salute You,Let's Get It Up,Rock
3,AC/DC,For Those About To Rock We Salute You,Inject The Venom,Rock
4,AC/DC,For Those About To Rock We Salute You,Snowballed,Rock
5,AC/DC,For Those About To Rock We Salute You,Evil Walks,Rock
6,AC/DC,For Those About To Rock We Salute You,C.O.D.,Rock
7,AC/DC,For Those About To Rock We Salute You,Breaking The Rules,Rock
8,AC/DC,For Those About To Rock We Salute You,Night Of The Long Knives,Rock
9,AC/DC,For Those About To Rock We Salute You,Spellbound,Rock


## Pandas: .loc vs. .iloc

When working with DataFrames in Pandas, there are two primary methods for selecting rows and columns: `.loc` and `.iloc`. It's crucial to understand the difference between them, as they work in fundamentally different ways.

### Key Differences:

| Feature | `.loc` | `.iloc` |
|---------|--------|--------|
| **Selection Method** | By **label** (index name) | By **integer position** |
| **Index Type** | Works with index labels (strings, dates, custom names) | Works with positions (0, 1, 2, ...) |
| **Includes End** | Inclusive (includes the end range) | Exclusive (excludes the end range) |
| **Syntax** | `.loc[row_label, column_label]` | `.iloc[row_position, column_position]` |

### .iloc: Integer Location
`.iloc` uses integer-based positioning, like Python lists. It's useful when you want to select rows or columns by their position.


In [10]:
# First, let's load the Track data again for demonstration
df = pd.read_csv('data/Track.csv', header=None)
df.columns = ["TrackId","Name","AlbumId","MediaTypeId","GenreId","Composer","Milliseconds","Bytes","UnitPrice"]


### .loc: Label-Based Location
`.loc` uses the index labels to select data. By default, the index is 0, 1, 2, etc., but `.loc` treats these as **labels**, not positions. Importantly, `.loc` is **inclusive on both ends** when using slices.


In [11]:
print("=== ILOC (exclusive on right end) ===")
print(df.iloc[1:4, [1, 5]])  # Positions 1 to 3 (4 is excluded)

print("\n=== LOC (inclusive on both ends) ===")
print(df.loc[1:4, ['Name', 'Composer']])  # Labels 1 to 4 (both included!)


=== ILOC (exclusive on right end) ===
                Name                                           Composer
1  Balls to the Wall                                                NaN
2    Fast As a Shark  F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...
3  Restless and Wild  F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...

=== LOC (inclusive on both ends) ===
                   Name                                           Composer
1     Balls to the Wall                                                NaN
2       Fast As a Shark  F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...
3     Restless and Wild  F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...
4  Princess of the Dawn                         Deaffy & R.A. Smith-Diesel


### When to Use Each

**Use `.iloc` when:**
- You want to select by position (like in lists)
- You know the row/column number
- You prefer Python-style exclusive end slicing

**Use `.loc` when:**
- You want to select by index label or column name
- You need to filter based on conditions (boolean indexing)
- You prefer inclusive slicing for clarity
- Working with custom indices (dates, strings, etc.)

**Best Practice:** `.loc` is generally preferred in modern Pandas code for its flexibility and readability, especially when combining filtering conditions.


### Important: Index Issues After Masking

When you use `.loc` to filter data with a boolean mask, the resulting DataFrame **retains the original index labels**. This can cause unexpected behavior or indexing errors if you later try to access rows by position or reset index-dependent operations.

**The Problem:**
- When you filter a DataFrame, the indices don't get renumbered (0, 1, 2, ...)
- Instead, the original indices are preserved
- This can cause confusion and errors when you try to work with the filtered data

**The Solution:**
Use `.reset_index(drop=True)` to renumber the rows from 0 to n-1 and discard the old index.


In [12]:
# Example: Notice the index preserves original row numbers after filtering
# First, find a composer that exists in the data
sample_composer = df['Composer'].dropna().iloc[0]
print(f"Filtering by composer: {sample_composer}")

filtered_tracks = df.loc[df['Composer'] == sample_composer, ['Name', 'Composer']]
print("\n--- Filtered data (WITH old indices) ---")
print(filtered_tracks.head())

print("\n--- After reset_index(drop=True) ---")
filtered_tracks_clean = filtered_tracks.reset_index(drop=True)
print(filtered_tracks_clean.head())

print("\n--- Now accessing by iloc works as expected ---")
print("First row:", filtered_tracks_clean.iloc[0]['Name'])


Filtering by composer: Angus Young, Malcolm Young, Brian Johnson

--- Filtered data (WITH old indices) ---
                                      Name  \
0  For Those About To Rock (We Salute You)   
5                    Put The Finger On You   
6                          Let's Get It Up   
7                         Inject The Venom   
8                               Snowballed   

                                    Composer  
0  Angus Young, Malcolm Young, Brian Johnson  
5  Angus Young, Malcolm Young, Brian Johnson  
6  Angus Young, Malcolm Young, Brian Johnson  
7  Angus Young, Malcolm Young, Brian Johnson  
8  Angus Young, Malcolm Young, Brian Johnson  

--- After reset_index(drop=True) ---
                                      Name  \
0  For Those About To Rock (We Salute You)   
1                    Put The Finger On You   
2                          Let's Get It Up   
3                         Inject The Venom   
4                               Snowballed   

                   